# Pro Kabaddi League (PKL) Analytics Platform
## 02: Comprehensive Exploratory Data Analysis (EDA)

**Objective:** Uncover key statistical patterns, trends, and tactical drivers across PKL matches, teams, players, and granular play-by-play events (Seasons 1–7).

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.analytics.metrics import (
    calculate_team_summary_metrics,
    calculate_player_summary_metrics,
    calculate_match_margin_metrics,
    calculate_event_summary_metrics
)

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print('Libraries and custom analytical modules imported successfully!')

### 1. Load Processed Analytical Tables
We read the clean Parquet tables generated by our Phase 2 data pipeline.

In [ ]:
df_matches = pd.read_parquet('data/processed/fact_matches.parquet')
df_team_match = pd.read_parquet('data/processed/fact_team_match.parquet')
df_player_match = pd.read_parquet('data/processed/fact_player_match.parquet')
df_events = pd.read_parquet('data/processed/fact_events.parquet')

print(f'Official Completed Matches: {len(df_matches[df_matches["is_official_pkl"] & df_matches["is_completed"]])}')
print(f'Total Team Records: {len(df_team_match)}')
print(f'Total Player Records: {len(df_player_match)}')
print(f'Total Event Plays: {len(df_events)}')

### 2. Match Scoring Distributions Across Seasons
Examine how total match scores and team scores have evolved across Seasons 1 to 7.

In [ ]:
df_tm_off = df_team_match[df_team_match['is_official_pkl'] & (df_team_match['status'] == 'Full Time')]
season_scores = df_tm_off.groupby('season_id')['score'].agg(['count', 'mean', 'std', 'min', 'median', 'max'])
season_scores.columns = ['Teams (x2)', 'Mean Score', 'Std Dev', 'Min Score', 'Median Score', 'Max Score']
display(season_scores.round(2))

# Display generated Figure 1
from IPython.display import Image
Image('reports/figures/fig1_scoring_by_season.png')

### 3. Match Outcomes & Winning Margins
Analyze the distribution of winning margins and quantify clutch finishes (<= 3 points).

In [ ]:
margin_stats = calculate_match_margin_metrics(df_matches)
print('Match Winning Margin Summary:')
for k, v in margin_stats.items():
    print(f'  {k:<25}: {v}')

Image('reports/figures/fig2_winning_margins.png')

### 4. Franchise Performance & Tactical Archetypes
Examine historical win percentages, point differentials, and raid vs tackle tactical quadrants.

In [ ]:
team_kpis = calculate_team_summary_metrics(df_team_match)
display(team_kpis[['team_name', 'matches_played', 'wins', 'losses', 'ties', 'win_percentage', 'point_differential', 'raid_success_rate', 'tackle_success_rate']].head(12))

Image('reports/figures/fig3_team_win_loss_records.png')

In [ ]:
# Tactical archetype quadrant
Image('reports/figures/fig4_raid_vs_tackle_scatter.png')

### 5. Player Intelligence: Legends of the League
Profile top raiders and top defenders by scoring efficiency, Super 10s, High 5s, and volatility.

In [ ]:
player_kpis = calculate_player_summary_metrics(df_player_match, min_matches=20)
print('Top 10 Raiders by Career Raid Points:')
display(player_kpis[player_kpis['player_role'] == 'Raider'][['player_name', 'appearances', 'total_raid_points', 'touch_points', 'bonus_points', 'super_10s', 'raid_strike_rate']].head(10))

print('\nTop 10 Defenders by Career Tackle Points:')
display(player_kpis[player_kpis['player_role'] == 'Defender'][['player_name', 'appearances', 'total_tackle_points', 'tackles_successful', 'high_5s', 'tackle_strike_rate']].head(10))

Image('reports/figures/fig5_top_raiders_defenders.png')

### 6. Play-by-Play Event Dynamics & Match Momentum
Break down raid outcome distributions and timeline score progression.

In [ ]:
event_stats = calculate_event_summary_metrics(df_events)
print('Event Stream Summary:')
for k, v in event_stats.items():
    if k != 'event_type_distribution':
        print(f'  {k:<25}: {v}')

Image('reports/figures/fig6_event_outcomes_breakdown.png')

In [ ]:
Image('reports/figures/fig8_score_progression_momentum.png')